# Regime Model 1st try

In [9]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from filterpy.kalman import KalmanFilter
import matplotlib.pyplot as plt
import logging
import glob
import os
import warnings
import feature_selection as fs

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class RegimeSwitchModel:
    def __init__(self, symbol, interval='1d', train_pct=0.8, strategy='long-only'):
        self.symbol = symbol
        self.interval = interval
        self.train_pct = train_pct
        if strategy not in ['long-only', 'long-short']:
            raise ValueError("Strategy must be either 'long-only' or 'long-short'")
        self.strategy = strategy
        self.data = None
        self.features = None
        self.selected_features = None
        self.train_data = None
        
        # Define paths for data
        self.onchain_data_path = "/Users/valter.rebelo/MissionControl/data/onchainData"
        self.macro_data_path = "/Users/valter.rebelo/MissionControl/data/macro/fredData"
        
        # Define macro features to use
        self.macro_features = [
            'treasury5YInflationExpectation', 
            'treasury5YInflationForwardRate', 
            'creditSpreads', 
            'vix', 
            'sp500', 
            'globalCbLiquidity'
        ]

    def load_data(self):
        """
        Load price data from files and prepare basic dataframe
        """
        try:
            # Load price data
            df_1 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/candleData/{self.symbol}_candles.csv")
            df_1['date'] = pd.to_datetime(df_1['date'])
            df_1.set_index('date', inplace=True)

            # Load market cap data
            df_2 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/assetData/{self.symbol}.csv")
            df_2['date'] = pd.to_datetime(df_2['date'])
            df_2.set_index('date', inplace=True)

            # Load BTC data for non-BTC assets
            btc_df = pd.read_csv("/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv")
            btc_df['date'] = pd.to_datetime(btc_df['date'])
            btc_df.set_index('date', inplace=True)

            # Merge price and market cap data
            data = pd.merge(df_1, df_2[['total_volume', 'market_cap']], on='date', how='inner')
            data.rename(columns={'total_volume': 'Volume', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'}, inplace=True)
            data.index.name = 'Date'

            # Add BTC relative price for non-BTC assets
            if self.symbol != "bitcoin":
                data['close_btc'] = (data['Close'] / btc_df['close']) * 100
                data.dropna(inplace=True)

            # Filter data for bitcoin
            if self.symbol == "bitcoin":
                data = data[data.index >= '2017-01-01']
            
            # Store the basic data
            self.data = data
            logging.info(f"Loaded {len(data)} rows of basic data for {self.symbol}")
            
            return data
            
        except Exception as e:
            logging.error(f"Data loading failed for {self.symbol}: {str(e)}")
            raise

    def generate_all_features(self):
        """
        Generate all features using feature_selection module with NO feature computation in notebook
        """
        if self.data is None:
            raise ValueError("Basic data must be loaded first. Call load_data() before generate_all_features().")
        
        try:
            
            # Call the feature generation function from feature_selection.py
            feature_results = fs.generate_features(
                data=self.data,
                symbol=self.symbol,
                include_technical=True,
                include_onchain=True,
                include_macro=True,
                onchain_data_path=self.onchain_data_path,
                macro_data_path=self.macro_data_path,
                macro_features=self.macro_features,
                verbose=True,
                show_plots=True  # Set to False to reduce logging noise
            )
            # Extract the processed data from the returned dictionary
            self.features = feature_results['processed_data']
            
            logging.info(f"Generated {len(self.features.columns)} features using feature_selection module")
            return self.features
        
        except Exception as e:
            logging.error(f"Feature generation failed: {str(e)}")
            raise

    def select_features(self, correlation_threshold=0.15, min_consensus=2, verbose=True):
        """
        Select the best features using feature selection from feature_selection module
        
        Args:
            correlation_threshold: Minimum absolute correlation with target
            min_consensus: Minimum number of methods that must select a feature
            verbose: Whether to print detailed information during selection
            
        Returns:
            List of selected feature names
        """
        if self.features is None:
            raise ValueError("Features must be generated first. Call generate_all_features() before select_features().")
        
        # Use the existing select_features function from the module
        results = fs.select_features(
            data=self.features,
            correlation_threshold=correlation_threshold,
            min_consensus=min_consensus,
            verbose=verbose
        )
        
        # Extract the recommended features from the results
        final_features = results['consensus_features']
        
        logging.info(f"Selected {len(final_features)} features using feature_selection module")
        
        # Validate we have enough features
        if len(final_features) < 3:
            logging.warning(f"Feature selection returned too few features: {final_features}. Using default features.")
            # Fallback to some default features if selection fails
            final_features = ['rsi_14']
        
        self.selected_features = final_features
        
        # Make sure we have log_close for Kalman Filter
        if 'log_close' not in self.features.columns:
            self.features['log_close'] = np.log(self.features['Close'])
        
        # Create model_data with log_return, log_close and selected features
        self.model_data = self.features[['log_return', 'log_close', 'Open', 'Close', 'Volume'] + self.selected_features]
        
        return self.selected_features

    def set_manual_features(self, feature_list):
        """
        Manually set the features to use for modeling, overriding automatic feature selection.
        
        Args:
            feature_list: List of feature names to use
            
        Returns:
            List of validated feature names that exist in the data
        """
        if self.features is None:
            raise ValueError("Features must be generated first. Call generate_all_features() before set_manual_features().")
        
        # Validate that the features exist in the data
        available_features = self.features.columns.tolist()
        valid_features = [f for f in feature_list if f in available_features]
        
        if len(valid_features) == 0:
            raise ValueError("None of the specified features exist in the data.")
        
        if len(valid_features) < len(feature_list):
            missing = set(feature_list) - set(valid_features)
            logging.warning(f"Some requested features are not available: {missing}")
        
        # Set the selected features
        self.selected_features = valid_features
        logging.info(f"Manually set {len(valid_features)} features: {valid_features}")
        
        # Make sure we have log_close for modeling
        if 'log_close' not in self.features.columns:
            self.features['log_close'] = np.log(self.features['Close'])
        
        # Create model_data with essential price columns, log_return, log_close and selected features
        essential_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        essential_available = [col for col in essential_columns if col in self.features.columns]
        
        # Ensure log_return is included
        columns_to_include = essential_available + ['log_return', 'log_close'] + self.selected_features
        # Remove duplicates while preserving order
        columns_to_include = list(dict.fromkeys(columns_to_include))
        
        self.model_data = self.features[columns_to_include]
        
        return valid_features

    # Keep the rest of the methods unchanged
    def split_data(self, data, embargo_percent=0.01):
        # Existing implementation
        try:
            total_rows = len(data)
            train_end_idx = int(total_rows * self.train_pct)
            embargo_end_idx = int(train_end_idx + (total_rows * embargo_percent))
            
            train = data.iloc[:train_end_idx]
            embargo = data.iloc[train_end_idx:embargo_end_idx]
            test = data.iloc[embargo_end_idx:]
            
            self.train_data = train

            if len(train) < 50 or len(test) < 50:
                raise ValueError("Train or test set too small (<50 rows).")
            logging.info(f"Split data: train={len(train)}, embargo={len(embargo)}, test={len(test)}")
            return train, embargo, test
        
        except Exception as e:
            logging.error(f"Data splitting failed: {str(e)}")
            raise

    def normalize_features(self, train, test, features):

        train_mean = train[features].mean()
        train_std = train[features].std()
        train_normalized = (train[features] - train_mean) / train_std
        test_normalized = (test[features] - train_mean) / train_std

        return (pd.DataFrame(train_normalized, index=train.index, columns=features),
                pd.DataFrame(test_normalized, index=test.index, columns=features))

    def train_hmm(self, train, features):
        try:
            f_train_normalized, _ = self.normalize_features(train, train, features)
            hmm = GaussianHMM(n_components=3, covariance_type='full', n_iter=5000, random_state=42)
            hmm.fit(f_train_normalized)
            if not hmm.monitor_.converged:
                logging.warning("HMM training did not converge.")
            logging.info("HMM trained successfully")
            return hmm
        except Exception as e:
            logging.error(f"HMM training failed: {str(e)}")
            raise

    def resample_weekly(self, data):
        """
        Resample data to weekly frequency with appropriate aggregation methods
        for different types of features.
        
        Args:
            data: DataFrame to resample
            
        Returns:
            Weekly resampled DataFrame
        """
        # Define standard aggregation methods for common columns
        agg_dict = {
            'Open': 'first', 
            'High': 'max', 
            'Low': 'min', 
            'Close': 'last',
            'Volume': 'sum', 
            'log_close': 'last', 
            'log_return': 'sum'
        }
        
        # Add dynamic aggregation for all other columns in the data
        for col in data.columns:
            if col not in agg_dict:
                # For technical indicators, use appropriate aggregation
                if any(x in col for x in ['rsi_', 'stoch_k', 'stoch_d', 'bb_', 'adx', 'zscore']):
                    agg_dict[col] = 'mean'
                # For moving averages
                elif any(x in col for x in ['sma_', 'ema_', 'ma_cross_', 'ma_distance_norm_']):
                    agg_dict[col] = 'last'
                # For momentum indicators
                elif any(x in col for x in ['macd', 'macd_signal', 'macd_diff', 'mom']):
                    agg_dict[col] = 'last'
                # For volatility indicators
                elif any(x in col for x in ['realized_vol_', 'high_low_range', 'atr_']):
                    agg_dict[col] = 'mean'
                # For volume indicators
                elif any(x in col for x in ['obv', 'volume_', 'vwap']):
                    agg_dict[col] = 'last'
                # For on-chain metrics
                elif any(x in col for x in ['mvrv', 'sopr', 'profit', 'loss', 'hash_rate', 'difficulty', 'active_addresses']):
                    agg_dict[col] = 'last'
                # For derived on-chain features
                elif any(x in col for x in ['_ma7', '_ma30', '_std30', '_mom7', '_mom30']):
                    agg_dict[col] = 'last'
                # For price ratios
                elif any(x in col for x in ['price_sma', 'price_ema']):
                    agg_dict[col] = 'last'
                # For everything else, use last as default
                else:
                    agg_dict[col] = 'last'
        
        # Only include columns that actually exist in the data
        valid_agg_dict = {col: method for col, method in agg_dict.items() if col in data.columns}
        
        # Perform resampling with the dynamic aggregation dictionary
        weekly = data.resample('W-MON').agg(valid_agg_dict)
        
        return weekly.dropna()

    def train_models_multi_res(self, train, test, features):
        """
        Train models at multiple time resolutions (daily and weekly)
        
        Args:
            train: Training data
            test: Test data
            features: List of features to use
            
        Returns:
            Tuple of model components for daily and weekly models
        """
        try:
            # Ensure all required features exist in the data
            # First, add log_close if it doesn't exist
            train_copy = train.copy()
            test_copy = test.copy()
            
            if 'log_close' not in train_copy.columns:
                if 'Close' in train_copy.columns:
                    train_copy['log_close'] = np.log(train_copy['Close'])
                    logging.info("Added log_close column to training data")
                else:
                    raise ValueError("Neither log_close nor Close columns exist in the data")
                    
            if 'log_close' not in test_copy.columns:
                if 'Close' in test_copy.columns:
                    test_copy['log_close'] = np.log(test_copy['Close'])
                    logging.info("Added log_close column to test data")
                else:
                    raise ValueError("Neither log_close nor Close columns exist in the data")
            
            # Check for missing features
            missing_cols = [col for col in features if col not in train_copy.columns]
            if missing_cols:
                logging.warning(f"Missing feature columns: {missing_cols}. Using available features only.")
                features = [f for f in features if f in train_copy.columns]
                
                if not features:
                    raise ValueError("No valid features available for modeling")
            
            # Daily models
            f_train_daily, f_test_daily = self.normalize_features(train_copy, test_copy, features)
            hmm_daily = self.train_hmm(train_copy, features)
            
            # Find optimal states for daily model based on Sharpe ratio
            daily_optimal_states = self.find_optimal_states(hmm_daily, train_copy, features)
            
            # Weekly models
            train_weekly = self.resample_weekly(train_copy)
            test_weekly = self.resample_weekly(test_copy)
            
            # Ensure features exist in weekly data
            weekly_features = [f for f in features if f in train_weekly.columns]
            if len(weekly_features) < len(features):
                logging.warning(f"Some features missing in weekly data. Using {len(weekly_features)} available features.")
            
            f_train_weekly, f_test_weekly = self.normalize_features(train_weekly, test_weekly, weekly_features)
            hmm_weekly = self.train_hmm(train_weekly, weekly_features)
            
            # Find optimal states for weekly model based on Sharpe ratio
            weekly_optimal_states = self.find_optimal_states(hmm_weekly, train_weekly, weekly_features)

            logging.info("Multi-resolution models trained successfully")
            
            return (hmm_daily, f_test_daily, daily_optimal_states), (hmm_weekly, f_test_weekly, weekly_optimal_states)
        except Exception as e:
            logging.error(f"Multi-resolution training failed: {str(e)}")
            raise

    def find_optimal_states(self, hmm, train_data, features):
        """
        Find optimal state mappings that maximize Sharpe ratio on training data
        
        Args:
            hmm: Trained HMM model
            train_data: Training data
            features: Features used for training
            
        Returns:
            Dictionary mapping state indices to trading actions ('Long', 'Short', 'Flat')
        """
        try:
            # Normalize features for prediction
            f_train_normalized, _ = self.normalize_features(train_data, train_data, features)
            
            # Predict states on training data
            hidden_states = hmm.predict(f_train_normalized)
            
            # Get unique states
            unique_states = np.unique(hidden_states)
            n_states = len(unique_states)
            
            # Calculate returns for each state
            state_returns = {}
            for state in unique_states:
                state_mask = (hidden_states == state)
                if sum(state_mask) > 0:
                    # Use log_return if available, otherwise calculate from Close
                    if 'log_return' in train_data.columns:
                        returns = train_data.loc[state_mask, 'log_return'].values
                    else:
                        returns = np.diff(np.log(train_data.loc[state_mask, 'Close'].values))
                        
                    state_returns[state] = {
                        'mean': np.mean(returns),
                        'std': np.std(returns) if len(returns) > 1 else 1e-6,  # Avoid division by zero
                        'sharpe': np.mean(returns) / (np.std(returns) if len(returns) > 1 else 1e-6),
                        'count': len(returns)
                    }
            
            # Determine optimal state mappings based on Sharpe ratio
            optimal_states = {}
            
            # Sort states by Sharpe ratio
            sorted_states = sorted(state_returns.items(), key=lambda x: x[1]['sharpe'], reverse=True)
            
            if self.strategy == 'long-only':
                # For long-only, the state with highest Sharpe is Long, others are Flat
                for i, (state, _) in enumerate(sorted_states):
                    if i == 0 and state_returns[state]['mean'] > 0:  # Only assign Long if mean return is positive
                        optimal_states[state] = 'Long'
                    else:
                        optimal_states[state] = 'Flat'
            else:  # long-short
                # For long-short, highest Sharpe is Long, lowest is Short, others are Flat
                for i, (state, metrics) in enumerate(sorted_states):
                    if i == 0 and metrics['mean'] > 0:  # Highest Sharpe and positive mean
                        optimal_states[state] = 'Long'
                    elif i == len(sorted_states) - 1 and metrics['mean'] < 0:  # Lowest Sharpe and negative mean
                        optimal_states[state] = 'Short'
                    else:
                        optimal_states[state] = 'Flat'
            
            # Log the optimal state mappings
            logging.info(f"Optimal state mappings based on Sharpe ratio: {optimal_states}")
            for state, action in optimal_states.items():
                if state in state_returns:
                    metrics = state_returns[state]
                    logging.info(f"State {state} -> {action}: Sharpe={metrics['sharpe']:.2f}, Mean={metrics['mean']:.4f}, Count={metrics['count']}")
            
            return optimal_states
        except Exception as e:
            logging.error(f"Optimal state finding failed: {str(e)}")
            # Fallback to default mapping based on means
            state_means = hmm.means_[:, 0]
            default_mapping = {}
            for state in range(len(state_means)):
                if state == np.argmax(state_means):
                    default_mapping[state] = 'Long'
                elif self.strategy == 'long-short' and state == np.argmin(state_means):
                    default_mapping[state] = 'Short'
                else:
                    default_mapping[state] = 'Flat'
            logging.warning(f"Using fallback state mapping: {default_mapping}")
            return default_mapping

    def voting_machine(self, hmm_daily, f_test_daily, daily_optimal_states, 
                    hmm_weekly, f_test_weekly, weekly_optimal_states, test_daily):
        
        # Daily predictions
        hidden_states_daily = hmm_daily.predict(f_test_daily)
        hmm_daily_state = [daily_optimal_states.get(s, 'Flat') for s in hidden_states_daily]
        
        logging.info(f"Daily predictions: {len(hmm_daily_state)} states")

        # Weekly predictions
        hidden_states_weekly = hmm_weekly.predict(f_test_weekly)
        hmm_weekly_state = [weekly_optimal_states.get(s, 'Flat') for s in hidden_states_weekly]

        weekly_df_raw = pd.DataFrame({
            'hmm_weekly': hmm_weekly_state
        }, index=f_test_weekly.index)
        weekly_df = weekly_df_raw.reindex(test_daily.index[1:], method='ffill')
        
        logging.info(f"Weekly predictions: {len(hmm_weekly_state)} states, reindexed to {len(weekly_df)} rows")

        if len(hmm_daily_state) != len(weekly_df):
            logging.warning(f"Length mismatch: daily={len(hmm_daily_state)}, weekly={len(weekly_df)}. Using minimum length.")

        ens_state = []
        min_len = min(len(hmm_daily_state), len(weekly_df))
        
        for i in range(min_len):
            daily_long = hmm_daily_state[i] == 'Long'
            daily_short = (self.strategy == 'long-short' and hmm_daily_state[i] == 'Short')
            
            weekly_long = weekly_df['hmm_weekly'].iloc[i] == 'Long'
            weekly_short = (self.strategy == 'long-short' and weekly_df['hmm_weekly'].iloc[i] == 'Short')
            
            if daily_long and weekly_long:
                ens_state.append('Long')
            elif daily_short and weekly_short:
                ens_state.append('Short')
            else:
                ens_state.append('Flat')
                
        return pd.Series(ens_state, index=test_daily.index[1:min_len+1])

    def print_voting_table(self, hmm_daily, f_test_daily, daily_optimal_states, 
                        hmm_weekly, f_test_weekly, weekly_optimal_states, test_daily):
        """
        Generate and print a simplified voting table showing how each model contributes
        to the final ensemble decision.
        
        Args:
            Same arguments as voting_machine method
            
        Returns:
            DataFrame containing the voting table
        """
        # Daily predictions
        hidden_states_daily = hmm_daily.predict(f_test_daily)
        hmm_daily_state = [daily_optimal_states.get(s, 'Flat') for s in hidden_states_daily]

        # Weekly predictions
        hidden_states_weekly = hmm_weekly.predict(f_test_weekly)
        hmm_weekly_state = [weekly_optimal_states.get(s, 'Flat') for s in hidden_states_weekly]

        weekly_df_raw = pd.DataFrame({
            'hmm_weekly': hmm_weekly_state
        }, index=f_test_weekly.index)
        weekly_df = weekly_df_raw.reindex(test_daily.index[1:], method='ffill')
        
        # Create the voting table
        min_len = min(len(hmm_daily_state), len(weekly_df))
        
        voting_table = pd.DataFrame({
            'Date': test_daily.index[1:min_len+1],
            'HMM Daily': hmm_daily_state[:min_len],
            'HMM Weekly': weekly_df['hmm_weekly'].iloc[:min_len].values,
        })
        
        # Add the ensemble decision
        ens_state = []
        for i in range(min_len):
            daily_long = hmm_daily_state[i] == 'Long'
            daily_short = (self.strategy == 'long-short' and hmm_daily_state[i] == 'Short')
            
            weekly_long = weekly_df['hmm_weekly'].iloc[i] == 'Long'
            weekly_short = (self.strategy == 'long-short' and weekly_df['hmm_weekly'].iloc[i] == 'Short')
            
            if daily_long and weekly_long:
                ens_state.append('Long')
            elif daily_short and weekly_short:
                ens_state.append('Short')
            else:
                ens_state.append('Flat')
        
        voting_table['Ensemble'] = ens_state
        voting_table.set_index('Date', inplace=True)
        self.voting_table = voting_table
        # Display the full table
        display(voting_table)
        
        # Calculate agreement statistics
        agreement_stats = {
            'Daily-Weekly Agreement': sum(hmm_daily_state[i] == weekly_df['hmm_weekly'].iloc[i] 
                                        for i in range(min_len)) / min_len,
            'Long Signals': sum(voting_table['Ensemble'] == 'Long') / min_len,
            'Short Signals': sum(voting_table['Ensemble'] == 'Short') / min_len,
            'Flat Signals': sum(voting_table['Ensemble'] == 'Flat') / min_len
        }
        
        print("\nModel Agreement Statistics:")
        for stat, value in agreement_stats.items():
            print(f"{stat}: {value:.2%}")
        
        return voting_table
    
    def plot_backtest(self, results):
        """
        Plot backtest results showing cumulative returns and market states.
        
        Args:
            results: DataFrame with backtest results
            
        Returns:
            Plotly figure object
        """
        from plotly.subplots import make_subplots
        import plotly.graph_objects as go
        
        # Create figure with secondary y-axis
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            vertical_spacing=0.1,
                            subplot_titles=(f"{self.symbol} Strategy vs Buy & Hold Returns", 
                                        "Market States"))

        # Add returns traces on first subplot
        fig.add_trace(go.Scatter(x=results.index, y=results['cum_returns'], 
                                mode='lines', name='Strategy Returns'),
                    row=1, col=1)
        fig.add_trace(go.Scatter(x=results.index, y=results['bnh_cum_returns'], 
                                mode='lines', name='Buy & Hold Returns'),
                    row=1, col=1)

        # Add states trace on second subplot
        states = results['ens_state']
        # Convert states to numeric for plotting
        state_map = {'Long': 1, 'Flat': 0, 'Short': -1}
        
        # Handle NaN values in states
        numeric_states = []
        for s in states:
            if pd.isna(s):
                numeric_states.append(None)  # Use None for NaN values
            else:
                numeric_states.append(state_map.get(s, 0))  # Default to 0 (Flat) for unknown states
        
        fig.add_trace(go.Scatter(x=states.index, y=numeric_states,
                                mode='lines', name='Market State',
                                hovertext=states),
                    row=2, col=1)

        # Update layout
        fig.update_layout(height=800,
                        showlegend=True)
        fig.update_yaxes(title_text="Cumulative Returns", row=1, col=1)
        fig.update_yaxes(title_text="State", row=2, col=1)
        fig.update_xaxes(title_text="Date", row=2, col=1)

        # Display the figure
        fig.show()
        
        return fig

    def ensemble_predict(self, test, features, show_voting_table=True):
        """
        Train models and make ensemble predictions.
        
        Args:
            test: Test data
            features: Features to use for prediction
            show_voting_table: Whether to display the voting table
            
        Returns:
            Series of predicted states
        """
        try:
            models_daily, models_weekly = self.train_models_multi_res(self.train_data, test, features)
            
            if show_voting_table:
                voting_table = self.print_voting_table(*models_daily, *models_weekly, test)
                self.voting_table = voting_table  # Store for later reference
            
            states = self.voting_machine(*models_daily, *models_weekly, test)
            logging.info("Ensemble prediction completed")
            return states
        except Exception as e:
            logging.error(f"Ensemble prediction failed: {str(e)}")
            raise


    def simulate_trading(self, test, states):
        try:
            # Strategy returns
            shifted_states = states.shift(1).fillna('Flat')
            position_multiplier = (shifted_states == 'Long').astype(int) - (shifted_states == 'Short').astype(int)
            returns = test['Open'].pct_change() * position_multiplier
            cum_returns = (1 + returns).cumprod()
            
            # Buy and hold returns
            bnh_returns = test['Close'].pct_change().fillna(0.0)
            bnh_cum_returns = (1 + bnh_returns).cumprod()
            
            result = pd.DataFrame({
                'Close': test['Close'], 
                'ens_state': states,
                'returns': returns, 
                'cum_returns': cum_returns,
                'bnh_returns': bnh_returns,
                'bnh_cum_returns': bnh_cum_returns
            })
            logging.info("Trading simulation completed")
            
            # Automatically plot the backtest results
            self.plot_backtest(result)
            
            return result
        except Exception as e:
            logging.error(f"Trading simulation failed: {str(e)}")
            raise

    def evaluate(self, results):
        try:
            # Strategy evaluation
            logging.info(f"ens_state sample: {results['ens_state'].head().tolist()}")
            cum_returns = results['cum_returns'].fillna(1.0)
            logging.info(f"Final cum_returns: {cum_returns.iloc[-1]}, length: {len(results)}")
            returns = results['returns'].fillna(0.0)
            
            # Calculate strategy metrics
            ann_ret = (cum_returns.iloc[-1] ** (365/len(results))) - 1
            sharpe = (returns.mean() / returns.std()) * np.sqrt(365)
            # Calculate Sortino ratio (using negative returns only for denominator)
            neg_returns = returns[returns < 0]
            sortino = (returns.mean() / neg_returns.std()) * np.sqrt(365) if len(neg_returns) > 0 else np.inf
            ann_vol = returns.std() * np.sqrt(365)
            drawdowns = cum_returns / cum_returns.cummax() - 1
            max_dd = drawdowns.min()
            
            # Count strategy switches
            ens_state = results['ens_state']
            switches = sum(ens_state.iloc[i] != ens_state.iloc[i-1] for i in range(1, len(ens_state)))
            
            # Buy and hold metrics
            bnh_returns = results['bnh_returns']
            bnh_cum_returns = results['bnh_cum_returns']
            bnh_ann_ret = (bnh_cum_returns.iloc[-1] ** (365/len(results))) - 1
            bnh_sharpe = (bnh_returns.mean() / bnh_returns.std()) * np.sqrt(365)
            # Calculate benchmark Sortino
            bnh_neg_returns = bnh_returns[bnh_returns < 0]
            bnh_sortino = (bnh_returns.mean() / bnh_neg_returns.std()) * np.sqrt(365) if len(bnh_neg_returns) > 0 else np.inf
            bnh_ann_vol = bnh_returns.std() * np.sqrt(365)
            bnh_drawdowns = bnh_cum_returns / bnh_cum_returns.cummax() - 1
            bnh_max_dd = bnh_drawdowns.min()
            
            # Create metrics table
            metrics_df = pd.DataFrame({
                'Metric': ['Annualized Return', 'Sharpe Ratio', 'Sortino Ratio', 'Annualized Volatility', 
                          'Maximum Drawdown', 'Final Cum Return', 'Switches'],
                'Strategy': [ann_ret, sharpe, sortino, ann_vol, max_dd, cum_returns.iloc[-1], switches],
                'Buy & Hold': [bnh_ann_ret, bnh_sharpe, bnh_sortino, bnh_ann_vol, bnh_max_dd, 
                              bnh_cum_returns.iloc[-1], 'N/A']
            })
            metrics_df.set_index('Metric', inplace=True)
            
            logging.info(f"Evaluation metrics:\n{metrics_df}")
            return metrics_df
        except Exception as e:
            logging.error(f"Evaluation failed: {str(e)}")
            raise



In [56]:
# Load data and generate features as usual
model = RegimeSwitchModel('bitcoin', strategy='long-short', train_pct=0.75)
data = model.load_data()
features = model.generate_all_features()


2025-03-06 17:28:21,708 - INFO - Loaded 2985 rows of basic data for bitcoin
2025-03-06 17:28:21,711 - INFO - Starting feature generation pipeline...
2025-03-06 17:28:21,714 - INFO - Generating technical indicators...
2025-03-06 17:28:21,821 - INFO - Loading and processing on-chain metrics...
2025-03-06 17:28:21,822 - INFO - Loading on-chain data for bitcoin...


2025-03-06 17:28:21,996 - INFO - Loaded 28 on-chain metrics
2025-03-06 17:28:22,036 - INFO - Processed 28 on-chain metrics out of 28 total
2025-03-06 17:28:22,037 - INFO - Loading and processing macro features...
2025-03-06 17:28:22,037 - INFO - Loading macro data...
2025-03-06 17:28:22,067 - INFO - Loaded 6 macro metrics
2025-03-06 17:28:22,104 - WARNING - Column treasury5YInflationExpectation has too many zero standard deviation values, skipping z-score calculation
2025-03-06 17:28:22,107 - WARNING - Column treasury5YInflationForwardRate has too many zero standard deviation values, skipping z-score calculation
2025-03-06 17:28:22,111 - WARNING - Column creditSpreads has too many zero standard deviation values, skipping z-score calculation
2025-03-06 17:28:22,114 - WARNING - Column vix has too many zero standard deviation values, skipping z-score calculation
2025-03-06 17:28:22,117 - WARNING - Column sp500 has too many zero standard deviation values, skipping z-score calculation
2025-

2025-03-06 17:28:22,600 - INFO - Generated 306 features using feature_selection module


In [57]:
selected_features = model.select_features(correlation_threshold=0.2, min_consensus=3)
selected_features



STARTING CONSENSUS FEATURE SELECTION
Input data shape: (2985, 306) (306 features)
Target column: log_return
Correlation threshold: 0.2
Minimum consensus required: 3 methods
--------------------------------------------------------------------------------

STEP 1: Correlation Filtering
--------------------------------------------------
Calculating correlations with log_return...
Range of correlations: -0.3273 to 0.6245
Mean absolute correlation: 0.0812
Features with positive correlation >= 0.2: 37
Features with negative correlation <= -0.2: 4

Top 5 positively correlated features:
  price_sma7_ratio: +0.6245
  rsi_7: +0.4459
  sth_sopr_zscore: +0.4315
  stoch_k: +0.4138
  sth_sopr: +0.3994

Top 5 negatively correlated features:
  utxo_loss_count_zscore: -0.3273
  relative_unrealized_loss_zscore: -0.3247
  relative_unrealized_loss_mom7: -0.2642
  utxo_loss_count_mom7: -0.2239

Features after correlation filtering (|correlation| >= 0.2): 41
Removed 264 features with low correlation

STEP 


Complete list of consensus features:


2025-03-06 17:28:36,230 - INFO - Selected 12 features using feature_selection module



Feature Selection Summary:
                                        correlation  consensus_count  lasso  \
price_sma7_ratio                           0.624502                3   True   
sth_sopr_zscore                            0.431484                3   True   
sth_sopr                                   0.399378                3   True   
mvrv_mom7                                  0.362772                3   True   
relative_unrealized_profit_mom7            0.357897                3   True   
rsi_14                                     0.344679                3   True   
entity_adj_nupl_mom7                       0.335569                3   True   
pct_supply_in_profit_zscore                0.328674                3   True   
relative_unrealized_loss_zscore            0.324662                3   True   
pct_supply_in_profit_mom7                  0.324499                3   True   
price_sma30_ratio                          0.318026                3   True   
ssr_oscillator          

['relative_unrealized_profit_mom7',
 'rsi_14',
 'pct_supply_in_profit_mom7',
 'mvrv_mom7',
 'price_sma30_ratio',
 'pct_supply_in_profit_zscore',
 'entity_adj_nupl_mom7',
 'price_sma7_ratio',
 'ssr_oscillator',
 'relative_unrealized_loss_zscore',
 'sth_sopr_zscore',
 'sth_sopr']

In [64]:
manual_features = ['relative_unrealized_profit_mom7',
 'rsi_14',
 'pct_supply_in_profit_mom7',
 'mvrv_mom7',
 'price_sma30_ratio',
 'pct_supply_in_profit_zscore',
 'entity_adj_nupl_mom7',
 'price_sma7_ratio',
 'relative_unrealized_loss_zscore',
 'sth_sopr_zscore',
 #### New Features ###
 #'mvrv_lth_zscore',
 
]

 
selected_features = model.set_manual_features(manual_features)

# Continue with the rest of your workflow as usual


2025-03-06 17:31:52,324 - INFO - Manually set 10 features: ['relative_unrealized_profit_mom7', 'rsi_14', 'pct_supply_in_profit_mom7', 'mvrv_mom7', 'price_sma30_ratio', 'pct_supply_in_profit_zscore', 'entity_adj_nupl_mom7', 'price_sma7_ratio', 'relative_unrealized_loss_zscore', 'sth_sopr_zscore']


In [65]:
# Split data into train, embargo, and test sets
train, embargo, test = model.split_data(model.model_data)

print(f"Train set: {len(train)} rows")
print(f"Embargo set: {len(embargo)} rows")
print(f"Test set: {len(test)} rows")

states = model.ensemble_predict(test, selected_features, show_voting_table=True)
results = model.simulate_trading(test, states)
metrics = model.evaluate(results)



2025-03-06 17:31:55,354 - INFO - Split data: train=2238, embargo=29, test=718


Train set: 2238 rows
Embargo set: 29 rows
Test set: 718 rows


2025-03-06 17:31:56,205 - INFO - HMM trained successfully
2025-03-06 17:31:56,210 - INFO - Optimal state mappings based on Sharpe ratio: {1: 'Long', 0: 'Flat', 2: 'Short'}
2025-03-06 17:31:56,211 - INFO - State 1 -> Long: Sharpe=0.11, Mean=0.0038, Count=970
2025-03-06 17:31:56,211 - INFO - State 0 -> Flat: Sharpe=0.01, Mean=0.0003, Count=754
2025-03-06 17:31:56,211 - INFO - State 2 -> Short: Sharpe=-0.02, Mean=-0.0012, Count=514
2025-03-06 17:31:56,260 - INFO - HMM trained successfully
2025-03-06 17:31:56,264 - INFO - Optimal state mappings based on Sharpe ratio: {2: 'Long', 1: 'Flat', 0: 'Short'}
2025-03-06 17:31:56,265 - INFO - State 2 -> Long: Sharpe=0.89, Mean=0.0802, Count=118
2025-03-06 17:31:56,265 - INFO - State 1 -> Flat: Sharpe=0.02, Mean=0.0010, Count=142
2025-03-06 17:31:56,265 - INFO - State 0 -> Short: Sharpe=-0.83, Mean=-0.1041, Count=61
2025-03-06 17:31:56,266 - INFO - Multi-resolution models trained successfully


,HMM Daily,HMM Weekly,Ensemble
Date,,,
2023-03-19,Long,NaN,Flat
2023-03-20,Short,Long,Flat
2023-03-21,Short,Long,Flat
2023-03-22,Long,Long,Long
2023-03-23,Long,Long,Long
...,...,...,...
2025-02-28,Short,Flat,Flat
2025-03-01,Short,Flat,Flat
2025-03-02,Long,Flat,Flat


2025-03-06 17:31:56,286 - INFO - Daily predictions: 718 states
2025-03-06 17:31:56,287 - INFO - Weekly predictions: 104 states, reindexed to 717 rows
2025-03-06 17:31:56,287 - WARNING - Length mismatch: daily=718, weekly=717. Using minimum length.
2025-03-06 17:31:56,293 - INFO - Ensemble prediction completed
2025-03-06 17:31:56,298 - INFO - Trading simulation completed



Model Agreement Statistics:
Daily-Weekly Agreement: 63.74%
Long Signals: 23.15%
Short Signals: 3.63%
Flat Signals: 73.22%


2025-03-06 17:31:57,086 - INFO - ens_state sample: [nan, 'Flat', 'Flat', 'Flat', 'Long']
2025-03-06 17:31:57,087 - INFO - Final cum_returns: 3.885518449223111, length: 718
2025-03-06 17:31:57,092 - INFO - Evaluation metrics:
                        Strategy Buy & Hold
Metric                                     
Annualized Return       0.993656   0.787749
Sharpe Ratio            2.507269   1.459038
Sortino Ratio           2.479124   2.282841
Annualized Volatility   0.292016   0.475176
Maximum Drawdown       -0.140208  -0.262319
Final Cum Return        3.885518   3.135581
Switches               57.000000        N/A


In [61]:
# Create an interactive time series plot for all columns in the voting table using Plotly
import plotly.graph_objects as go
import pandas as pd

# Convert categorical data to numeric for plotting
numeric_voting_table = pd.DataFrame()
for column in model.voting_table.columns:
    numeric_voting_table[column] = model.voting_table[column].map({'Long': 1, 'Short': -1, 'Flat': 0})

# Create figure
fig = go.Figure()

# Add traces for each column
for column in numeric_voting_table.columns:
    fig.add_trace(go.Scatter(
        x=numeric_voting_table.index,
        y=numeric_voting_table[column],
        mode='markers',  # Use dots instead of lines
        marker=dict(size=8),
        name=column,
        hovertemplate='%{x}<br>%{text}',
        text=[model.voting_table[column][i] for i in numeric_voting_table.index]  # Show original state in hover
    ))

# Add horizontal lines to indicate states
fig.add_shape(type="line", x0=numeric_voting_table.index[0], y0=1, x1=numeric_voting_table.index[-1], y1=1,
              line=dict(color="green", width=1, dash="dash"), name="Long")
fig.add_shape(type="line", x0=numeric_voting_table.index[0], y0=0, x1=numeric_voting_table.index[-1], y1=0,
              line=dict(color="gray", width=1, dash="dash"), name="Flat")
fig.add_shape(type="line", x0=numeric_voting_table.index[0], y0=-1, x1=numeric_voting_table.index[-1], y1=-1,
              line=dict(color="red", width=1, dash="dash"), name="Short")

# Update layout
fig.update_layout(
    title='Market States Over Time by Model',
    xaxis_title='Date',
    yaxis_title='Market State',
    yaxis=dict(
        tickvals=[-1, 0, 1],
        ticktext=['Short', 'Flat', 'Long']
    ),
    hovermode='closest',
    height=600,
    width=1000
)

# Show the plot
fig.show()